In [13]:
!pip install catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 8.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')

In [3]:
# 1. 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
# 2. 물리 법칙 기반 피처 엔지니어링 (핵심)
def physics_engineering(df):
    d = df.copy()

    # 기본 변환
    d["height_cm"] = (d["Height(Feet)"] * 12 + d["Height(Remainder_Inches)"]) * 2.54
    d["weight_kg"] = d["Weight(lb)"] * 0.453592
    d["temp_c"] = (d["Body_Temperature(F)"] - 32) * (5/9)

    # [핵심] 칼로리 공식 역추적 피처
    # 칼로리 ~= 시간 * 체중 * 강도
    # 이 조합들이 선형회귀에서 계수가 딱 떨어지게 만들어줌
    d['Formula_1'] = d['Exercise_Duration'] * d['weight_kg'] * d['BPM']
    d['Formula_2'] = d['Exercise_Duration'] * d['BPM']
    d['Formula_3'] = d['Exercise_Duration'] * d['weight_kg']

    # 비선형 보정
    d['Log_Formula_1'] = np.log1p(d['Formula_1'])

    # 성별 인코딩
    d['Gender'] = d['Gender'].apply(lambda x: 1 if x in ['M', 'Male'] else 0)
    d['Weight_Status'] = pd.factorize(d['Weight_Status'])[0]

    return d

train_df = physics_engineering(train)
test_df = physics_engineering(test)

X = train_df.drop(['ID', 'Calories_Burned'], axis=1)
y = train_df['Calories_Burned']
test_X = test_df.drop(['ID'], axis=1)

In [5]:
# 3. Pseudo Labeling을 위한 1차 학습
# 앙상블로 신뢰도 높은 예측값 생성
models = {
    'cat': CatBoostRegressor(iterations=3000, learning_rate=0.05, depth=8, random_seed=42, verbose=0, loss_function='RMSE'),
    'xgb': XGBRegressor(n_estimators=3000, learning_rate=0.05, max_depth=8, random_state=42, n_jobs=-1),
    'lgbm': LGBMRegressor(n_estimators=3000, learning_rate=0.05, num_leaves=100, random_state=42, n_jobs=-1, verbose=-1)
}

print(">>> 1차 예측 (Pseudo Labeling용) 진행 중...")
preds_1st = np.zeros(len(test_X))
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    fold_preds = np.zeros(len(test_X))
    for train_idx, val_idx in kf.split(X, y):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        fold_preds += model.predict(test_X) / 5
    preds_1st += fold_preds / 3

>>> 1차 예측 (Pseudo Labeling용) 진행 중...


In [6]:
# 4. Pseudo Labeling 적용
# 테스트 데이터 전체를 학습 데이터에 합침 (Soft Labeling)
X_pseudo = pd.concat([X, test_X], axis=0).reset_index(drop=True)
y_pseudo = pd.concat([y, pd.Series(preds_1st)], axis=0).reset_index(drop=True)

print(f">>> Pseudo Labeling 완료. 데이터 크기 증가: {len(X)} -> {len(X_pseudo)}")

>>> Pseudo Labeling 완료. 데이터 크기 증가: 7500 -> 15000


In [7]:
# 5. 최종 학습 (Stacking with Linear Meta-Model)
# 칼로리 예측은 수식에 가까우므로, 최종단은 LinearRegression이 유리할 수 있음
print(">>> 최종 Stacking 학습 진행 중...")

# 2차 모델 정의 (조금 더 깊게)
final_models = [
    CatBoostRegressor(iterations=5000, learning_rate=0.01, depth=9, random_seed=42, verbose=0, loss_function='RMSE'),
    XGBRegressor(n_estimators=5000, learning_rate=0.01, max_depth=9, random_state=42, n_jobs=-1),
    Ridge(alpha=0.5) # 선형 관계 포착용
]

meta_train = np.zeros((len(X_pseudo), len(final_models)))
meta_test = np.zeros((len(test_X), len(final_models)))

# Stacking Loop
for i, model in enumerate(final_models):
    kf_final = KFold(n_splits=5, shuffle=True, random_state=99)
    for train_idx, val_idx in kf_final.split(X_pseudo, y_pseudo):
        X_tr, X_val = X_pseudo.iloc[train_idx], X_pseudo.iloc[val_idx]
        y_tr, y_val = y_pseudo.iloc[train_idx], y_pseudo.iloc[val_idx]

        model.fit(X_tr, y_tr)
        meta_train[val_idx, i] = model.predict(X_val)

    model.fit(X_pseudo, y_pseudo)
    meta_test[:, i] = model.predict(test_X)

# 메타 모델 (Linear Regression for Formula Finding)
meta_learner = LinearRegression() # 중요! 0.0x를 위해선 Ridge보다 순수 Linear가 나을 수 있음
meta_learner.fit(meta_train, y_pseudo)
final_pred = meta_learner.predict(meta_test)

>>> 최종 Stacking 학습 진행 중...


In [8]:
# 6. [핵심 치트키] 후처리 (Post-Processing)
# 정수형 타겟에 맞춤 (166.0, 33.0 등)
final_pred_rounded = np.round(final_pred)

print("\n>>> 결과 확인")
print(f"Raw Prediction Sample: {final_pred[:5]}")
print(f"Rounded Prediction Sample: {final_pred_rounded[:5]}")


>>> 결과 확인
Raw Prediction Sample: [173.35480497 191.08047215  53.53752897 161.51630496 225.29704934]
Rounded Prediction Sample: [173. 191.  54. 162. 225.]


In [9]:

# 7. 저장
submission = pd.read_csv('sample_submission.csv')
submission['Calories_Burned'] = final_pred_rounded
submission.to_csv('./submit.csv', index = False)
print("submission_cheat_code.csv 저장 완료! (RMSE 0.0x 도전)")

submission_cheat_code.csv 저장 완료! (RMSE 0.0x 도전)


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 1. 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. 극한의 수식 피처 생성
def create_features(df):
    d = df.copy()
    d["weight_kg"] = d["Weight(lb)"] * 0.453592
    # 칼로리 소모의 핵심은 (운동시간 * 심박수 * 체중)의 상호작용입니다.
    d['Magic_Formula'] = d['Exercise_Duration'] * d['BPM'] * d['weight_kg']
    d['Intensity_Duration'] = d['BPM'] * d['Exercise_Duration']
    d['Gender'] = d['Gender'].map({'F':0, 'M':1, 'Female':0, 'Male':1})
    d['Weight_Status'] = pd.factorize(d['Weight_Status'])[0]

    # 불필요 컬럼 제거 (ID 등)
    drop_cols = ['ID', 'Height(Feet)', 'Height(Remainder_Inches)', 'Body_Temperature(F)']
    d = d.drop(columns=[c for c in drop_cols if c in d.columns])
    return d

X = create_features(train).drop(columns=['Calories_Burned'])
y = train['Calories_Burned']
X_test = create_features(test)

# 3. 교차 검증 및 RMSE 확인 루프
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
final_test_preds = np.zeros(len(X_test))

print("🚀 RMSE 0.0x 도전을 위한 최적화 학습 시작...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # 0.0x를 위해선 CatBoost가 가장 유리합니다.
    model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.01,
        depth=8,
        l2_leaf_reg=3,
        random_seed=42,
        verbose=0 # 출력 최적화
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)

    # 검증 데이터 예측
    val_pred = model.predict(X_val)

    # [치트키] 타겟이 정수이므로 반올림 처리
    val_pred = np.round(val_pred)

    oof_preds[val_idx] = val_pred
    final_test_preds += model.predict(X_test) / kf.n_splits

    fold_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"✅ Fold {fold+1} RMSE: {fold_rmse:.5f}")

# 4. 전체 검증 점수 출력 (이게 0.0x 여야 함)
total_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print("\n" + "="*30)
print(f"🔥 FINAL VALIDATION RMSE: {total_rmse:.5f}")
print("="*30)

# 5. 제출 파일 저장
submission = pd.read_csv('sample_submission.csv')
submission['Calories_Burned'] = np.round(final_pred_rounded) # 반올림
submission.to_csv('./submit.csv', index = False)

🚀 RMSE 0.0x 도전을 위한 최적화 학습 시작...
✅ Fold 1 RMSE: 1.29692
✅ Fold 2 RMSE: 0.84341
✅ Fold 3 RMSE: 0.81322
✅ Fold 4 RMSE: 0.84735
✅ Fold 5 RMSE: 0.97434

🔥 FINAL VALIDATION RMSE: 0.97180


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# 1. 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. 수식 복원을 위한 피처 엔지니어링
def final_engineering(df):
    d = df.copy()
    d["weight_kg"] = d["Weight(lb)"] * 0.45359237
    d["height_cm"] = (d["Height(Feet)"] * 12 + d["Height(Remainder_Inches)"]) * 2.54

    # [핵심] 칼로리 공식은 대부분 곱셈과 나눗셈으로 이루어짐
    # 모델이 곱셈을 쉽게 이해하도록 모든 수치형 변수의 Log값을 추가
    num_cols = ['Exercise_Duration', 'BPM', 'weight_kg', 'Age', 'Body_Temperature(F)']
    for col in num_cols:
        d[f'log_{col}'] = np.log1p(d[col])

    # 강력한 상호작용 피처 (물리 공식 기반)
    d['Intensity_Work'] = d['Exercise_Duration'] * d['BPM'] * d['weight_kg']
    d['Heat_Energy'] = d['Body_Temperature(F)'] * d['Exercise_Duration']

    # 범주형 처리
    d['Gender'] = d['Gender'].map({'F':0, 'M':1, 'Female':0, 'Male':1})
    d['Weight_Status'] = pd.factorize(d['Weight_Status'])[0]

    return d

X = final_engineering(train).drop(columns=['ID', 'Calories_Burned'])
y = train['Calories_Burned']
X_test = final_engineering(test).drop(columns=['ID'])

# 3. [비기] 타겟 로그 변환 (RMSE 0.0x의 핵심)
# 타겟값이 0~300으로 범위가 넓으므로 로그를 씌워 오차의 폭을 줄입니다.
y_log = np.log1p(y)

# 4. 극한의 파라미터 셋팅
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_log_preds = np.zeros(len(X))
test_log_preds = np.zeros(len(X_test))

print("🔥 RMSE 0.0x 진입을 위한 극한 학습 시작...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_log.iloc[train_idx], y_log.iloc[val_idx]

    # 극도로 낮은 learning_rate와 많은 iterations로 미세 오차 제거
    model = CatBoostRegressor(
        iterations=10000,        # 반복 횟수 대폭 증가
        learning_rate=0.005,    # 아주 천천히 학습 (정교함 상승)
        depth=10,               # 트리 깊이 확장
        l2_leaf_reg=2,
        random_seed=42,
        verbose=1000,
        loss_function='RMSE'
    )

    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=500)

    oof_log_preds[val_idx] = model.predict(X_val)
    test_log_preds += model.predict(X_test) / kf.n_splits

# 5. 역변환 및 후처리
# 로그를 다시 숫자로 돌린 후 정수 반올림
final_oof = np.expm1(oof_log_preds)
final_oof = np.round(final_oof)

total_rmse = np.sqrt(mean_squared_error(y, final_oof))
print("\n" + "="*40)
print(f"✨ FINAL VALIDATION RMSE: {total_rmse:.6f}")
print("="*40)

# 6. 결과 저장
submission = pd.read_csv('sample_submission.csv')
submission['Calories_Burned'] = np.round(np.expm1(test_log_preds))
submission.to_csv('./submit.csv', index = False)

🔥 RMSE 0.0x 진입을 위한 극한 학습 시작...
0:	learn: 0.9623177	test: 0.9357821	best: 0.9357821 (0)	total: 61.3ms	remaining: 10m 13s
1000:	learn: 0.0378526	test: 0.0456377	best: 0.0456377 (1000)	total: 19.1s	remaining: 2m 51s
2000:	learn: 0.0193360	test: 0.0308165	best: 0.0308165 (2000)	total: 36.1s	remaining: 2m 24s
3000:	learn: 0.0135582	test: 0.0269250	best: 0.0269250 (3000)	total: 53.8s	remaining: 2m 5s
4000:	learn: 0.0105970	test: 0.0253759	best: 0.0253759 (4000)	total: 1m 13s	remaining: 1m 50s
5000:	learn: 0.0088374	test: 0.0246938	best: 0.0246938 (5000)	total: 1m 29s	remaining: 1m 29s
6000:	learn: 0.0075468	test: 0.0243279	best: 0.0243279 (6000)	total: 1m 47s	remaining: 1m 11s
7000:	learn: 0.0065971	test: 0.0241309	best: 0.0241304 (6994)	total: 2m 3s	remaining: 53s
8000:	learn: 0.0058323	test: 0.0240066	best: 0.0240066 (8000)	total: 2m 22s	remaining: 35.6s
9000:	learn: 0.0052433	test: 0.0239309	best: 0.0239309 (9000)	total: 2m 39s	remaining: 17.7s
9999:	learn: 0.0047613	test: 0.0238734	best:

In [14]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor

# 1. 데이터 로드 및 초기 전처리
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

def get_extreme_features(df):
    d = df.copy()
    # 기본 단위 정밀 변환
    d["weight_kg"] = d["Weight(lb)"] * 0.45359237
    d["height_cm"] = (d["Height(Feet)"] * 12 + d["Height(Remainder_Inches)"]) * 2.54
    d["temp_c"] = (d["Body_Temperature(F)"] - 32) * (5/9)

    # [핵심] 상호작용 피처 폭발 (200개 이상 생성)
    core = ['Exercise_Duration', 'BPM', 'weight_kg', 'Age', 'temp_c']
    for i, c1 in enumerate(core):
        for c2 in core[i+1:]:
            d[f'{c1}_mul_{c2}'] = d[c1] * d[c2]
            d[f'{c1}_div_{c2}'] = d[c1] / (d[c2] + 1e-9)

    # 범주형 수치화
    d['Gender'] = d['Gender'].map({'F':0, 'M':1, 'Female':0, 'Male':1})
    d['Weight_Status'] = pd.factorize(d['Weight_Status'])[0]
    return d

X = get_extreme_features(train).drop(columns=['ID', 'Calories_Burned'])
y = train['Calories_Burned']
X_test = get_extreme_features(test).drop(columns=['ID'])

# 2. Ridge를 통한 Feature Selection (노이즈 제거)
selector = Ridge(alpha=1.0)
selector.fit(X, y)
importance = np.abs(selector.coef_)
top_features = X.columns[importance > np.percentile(importance, 20)] # 하위 20% 피처 제거
X = X[top_features]
X_test = X_test[top_features]

# 3. Seed Averaging + K-Fold (10 Seeds x 5 Folds = 50 Models)
seeds = [42, 123, 777, 2024, 999, 10, 55, 88, 1004, 21]
total_oof = np.zeros(len(X))
total_test = np.zeros(len(X_test))

print(f"🚀 총 {len(seeds)}개의 시드로 학습을 시작합니다. (피처 수: {X.shape[1]})")

for s_idx, seed in enumerate(seeds):
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X))

    for fold, (t_idx, v_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X.iloc[t_idx], X.iloc[v_idx]
        y_tr, y_val = y.iloc[t_idx], y.iloc[v_idx]

        model = CatBoostRegressor(
            iterations=8000,
            learning_rate=0.008,
            depth=9,
            l2_leaf_reg=3,
            random_seed=seed,
            verbose=0,
            early_stopping_rounds=200
        )

        model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

        seed_oof[v_idx] = model.predict(X_val)
        total_test += model.predict(X_test) / (len(seeds) * 5)

    total_oof += seed_oof / len(seeds)
    fold_rmse = np.sqrt(mean_squared_error(y, total_oof * (len(seeds)/(s_idx+1))))
    print(f"✅ Seed {seed} 완료 - 현재 누적 RMSE: {fold_rmse:.6f}")

# 4. 최종 후처리 (0.0x를 위한 마법)
# 칼로리는 정수일 확률이 매우 높으므로 반올림 적용
final_oof = np.round(total_oof)
final_test = np.round(total_test)

final_rmse = np.sqrt(mean_squared_error(y, final_oof))
print("\n" + "="*40)
print(f"🔥 FINAL RMSE (Rounded): {final_rmse:.6f}")
print("="*40)

# 5. 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
submission['Calories_Burned'] = final_test
submission.to_csv('./submit.csv', index = False)

🚀 총 10개의 시드로 학습을 시작합니다. (피처 수: 25)
✅ Seed 42 완료 - 현재 누적 RMSE: 1.123471
✅ Seed 123 완료 - 현재 누적 RMSE: 1.047265
✅ Seed 777 완료 - 현재 누적 RMSE: 1.032385
✅ Seed 2024 완료 - 현재 누적 RMSE: 1.026859
✅ Seed 999 완료 - 현재 누적 RMSE: 1.014075
✅ Seed 10 완료 - 현재 누적 RMSE: 1.023638
✅ Seed 55 완료 - 현재 누적 RMSE: 1.018578
✅ Seed 88 완료 - 현재 누적 RMSE: 1.022087
✅ Seed 1004 완료 - 현재 누적 RMSE: 1.026626
✅ Seed 21 완료 - 현재 누적 RMSE: 1.022910

🔥 FINAL RMSE (Rounded): 1.045307
